# NeXo v3.0 · Notebook 04 — RAT Underservice (XGBoost binary classifier)

> **CRISP-DM Phase 4 — Modeling** · Supervised binary classification.
> **Input:** `curated/subscribers.parquet` + `splits.json` (from nb 01).
> **Output:** `models/rat_underservice_v3_xgb.joblib` + `rat_v3_feature_names.joblib` + model card.
> **Consumed by:** `services/ai-service/model_cache.py` (mtime hot-reload) → `POST /infer/rat-underservice`.

## Pipeline
```
subscribers.parquet → label (rat_gap_score > threshold) → numeric features
            → XGBoost (scale_pos_weight for imbalance) → eval on random + temporal splits
            → save .joblib + feature names + model card
```

## Goal
Flag subscribers who are **4G-capable but stuck on 2G/3G traffic** → "underservice". The model
outputs a probability ∈ [0,1]; downstream playbooks (`pb-churn-prevention`) act on high scores.

## Why XGBoost
- Tabular gold-standard — boosted trees beat deep nets on structured features.
- GPU-accelerated (`device='cuda'`, `tree_method='hist'`) with automatic CPU fallback.
- Built-in class-imbalance handling via `scale_pos_weight`.

## Two evaluation regimes (both reported)
- **Random split** — best-case metric.
- **Temporal hold-out** — train on early months, test on the last → production realism (drift check).

## What this notebook does NOT do
- Does NOT compute the CEM score (nb 02) or OSS anomalies (nb 03).
- Does NOT define `rat_gap_score` — that's a feature from nb 01; here we only threshold it into a label.
- Does NOT deploy — writes the joblib artifact ai-service hot-loads.


## 1 · Imports + MinIO load

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from IPython.display import display

import io, json, os
from pathlib import Path
import boto3, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import xgboost as xgb
from botocore.client import Config
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_recall_curve, auc
sns.set_theme(style='whitegrid')
s3 = boto3.client('s3', endpoint_url=os.environ.get('S3_ENDPOINT','http://localhost:9000'),
                  aws_access_key_id='minio', aws_secret_access_key='minio_pw',
                  config=Config(signature_version='s3v4'))
subs = pd.read_parquet(io.BytesIO(s3.get_object(Bucket='curated', Key='subscribers.parquet')['Body'].read()))
splits = json.loads(s3.get_object(Bucket='curated', Key='splits.json')['Body'].read())
print(f'subs: {len(subs):,} rows  splits: {splits["meta"]["random_sizes"]}')

In [ ]:
# ── Parameters (papermill-overridable) ──────────────────────────────────────
# This cell is tagged `parameters`. retrain-service (papermill) can override any
# value without editing the notebook. EVERY magic number lives here.
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Label definition
RAT_LABEL_THRESHOLD = 0.30   # subscriber flagged underserved if rat_gap_score > this

# XGBoost hyperparameters
XGB_N_ESTIMATORS  = 500      # max boosting rounds
XGB_MAX_DEPTH     = 8        # tree depth (interaction order)
XGB_LEARNING_RATE = 0.07     # shrinkage per round
XGB_OBJECTIVE     = 'binary:logistic'
XGB_EVAL_METRIC   = 'auc'
XGB_TREE_METHOD   = 'hist'   # histogram split-finding (GPU-friendly)

# Decision + reporting
DECISION_THRESHOLD   = 0.50  # probability cutoff for the F1 operating point
TOP_K_IMPORTANCE     = 20     # features shown in the importance chart

# Artifact filenames — IMMUTABLE (services/ai-service/model_cache.py reads these)
RAT_MODEL_FNAME  = 'rat_underservice_v3_xgb.joblib'
RAT_FEATS_FNAME  = 'rat_v3_feature_names.joblib'
RAT_CARD_FNAME   = 'rat_underservice_v3_model_card.md'

print(f'SEED={SEED} | label>{RAT_LABEL_THRESHOLD} | xgb={XGB_N_ESTIMATORS}t/depth{XGB_MAX_DEPTH}/'
      f'lr{XGB_LEARNING_RATE} | thr={DECISION_THRESHOLD}')


## 2 · Build label + select features

In [ ]:
y = (subs['rat_gap_score'] > RAT_LABEL_THRESHOLD).astype(int)
print(f'positive rate: {y.mean()*100:.2f}%')
DROP = {'imsi','imsi_hash','rat_gap_score','cem_score_target','churn_risk_flag',
        'source_origin','source_file','_strata','month_year'}
feat_cols = [c for c in subs.columns if c not in DROP and pd.api.types.is_numeric_dtype(subs[c])]
X = subs[feat_cols].fillna(0)
print(f'features: {len(feat_cols)}')

## 3 · Split + class weight

In [ ]:
tr,va,te = splits['random']['train'], splits['random']['val'], splits['random']['test']
X_tr, y_tr = X.iloc[tr], y.iloc[tr]
X_va, y_va = X.iloc[va], y.iloc[va]
X_te, y_te = X.iloc[te], y.iloc[te]
neg = (y_tr==0).sum(); pos = (y_tr==1).sum()
spw = neg/max(pos,1)
print(f'class weight pos:neg = 1:{spw:.2f}')

## 4 · Train XGBoost (GPU if available)

500 estimators × depth 8 × lr 0.07 (matches `docs/architecture/ml-models.md`).

In [ ]:
params = dict(n_estimators=XGB_N_ESTIMATORS, max_depth=XGB_MAX_DEPTH,
              learning_rate=XGB_LEARNING_RATE,
              objective=XGB_OBJECTIVE, eval_metric=XGB_EVAL_METRIC,
              scale_pos_weight=spw, random_state=SEED, tree_method=XGB_TREE_METHOD)
model = xgb.XGBClassifier(**params)
try:
    model.set_params(device='cuda')
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
except Exception as e:
    print(f'GPU failed ({e}), CPU fallback'); model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
print('done')

## 5 · Evaluate test (random split)

In [ ]:
pte = model.predict_proba(X_te)[:,1]
m_random = {
    'ROC_AUC': float(roc_auc_score(y_te, pte)),
    'F1@0.5': float(f1_score(y_te, (pte>DECISION_THRESHOLD).astype(int))),
}
print('random:', json.dumps(m_random, indent=2))

## 6 · Evaluate temporal hold-out (production realism)

In [ ]:
ho = splits['temporal']['holdout_indices']
X_ho, y_ho = X.iloc[ho], y.iloc[ho]
pho = model.predict_proba(X_ho)[:,1]
m_temp = {
    'ROC_AUC': float(roc_auc_score(y_ho, pho)),
    'F1@0.5': float(f1_score(y_ho, (pho>DECISION_THRESHOLD).astype(int))),
    'holdout_size': int(len(ho)),
    'months': splits['temporal']['holdout_months'],
}
print('temporal:', json.dumps(m_temp, indent=2))

## 7 · Feature importance

In [ ]:
imp = pd.Series(model.feature_importances_, index=feat_cols).sort_values(ascending=False).head(TOP_K_IMPORTANCE)
fig, ax = plt.subplots(figsize=(8,7))
sns.barplot(x=imp.values, y=imp.index, palette='magma', orient='h', ax=ax)
ax.set_title('RAT — top 20 features'); plt.tight_layout(); plt.show()

## 8 · Save model + push to MinIO

In [ ]:
MODELS = Path('models'); MODELS.mkdir(exist_ok=True)
joblib.dump(model, MODELS/RAT_MODEL_FNAME)
joblib.dump(feat_cols, MODELS/RAT_FEATS_FNAME)
card = (f'# RAT Underservice v3 Model Card\n\n'
        f'## Random split\n- ROC-AUC: {m_random["ROC_AUC"]:.4f}\n- F1@0.5: {m_random["F1@0.5"]:.4f}\n\n'
        f'## Temporal hold-out\n- ROC-AUC: {m_temp["ROC_AUC"]:.4f}\n- F1@0.5: {m_temp["F1@0.5"]:.4f}\n')
(MODELS/RAT_CARD_FNAME).write_text(card); print(card)
for fn in [RAT_MODEL_FNAME, RAT_FEATS_FNAME, RAT_CARD_FNAME]:
    s3.put_object(Bucket='curated', Key=f'models/{fn}', Body=(MODELS/fn).read_bytes())
    print(f'  ↑ curated/models/{fn}')

---
## Done · register